<a href="https://colab.research.google.com/github/BhavanaVuyyuru/rag-document-search/blob/main/notebooks/rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install sentence-transformers -q
!pip install pandas numpy -q
print("✅ Done!")

✅ Done!


In [15]:
import pandas as pd
import numpy as np
import os

os.makedirs("data", exist_ok=True)
os.makedirs("output", exist_ok=True)

documents = [
    {"doc_id": "D001", "category": "diabetes",      "text": "Patient diagnosed with Type 2 diabetes. Prescribed metformin 500mg twice daily. Blood sugar at 180 mg/dL."},
    {"doc_id": "D002", "category": "cardiology",    "text": "Patient reports chest pain and shortness of breath. ECG shows irregular heartbeat. Referred to cardiologist."},
    {"doc_id": "D003", "category": "hypertension",  "text": "Blood pressure 145/92 mmHg. Stage 2 hypertension. Prescribed lisinopril 10mg and low sodium diet."},
    {"doc_id": "D004", "category": "diabetes",      "text": "Follow-up for diabetes. HbA1c improved from 9.2 to 7.8. Added insulin glargine for better glucose control."},
    {"doc_id": "D005", "category": "general",       "text": "Annual wellness checkup completed. Cholesterol 195 mg/dL normal. BMI 24.3. No abnormalities detected."},
    {"doc_id": "D006", "category": "cardiology",    "text": "Post cardiac surgery follow-up. Wound healing well. Patient advised to continue blood thinners and cardiac rehab."},
    {"doc_id": "D007", "category": "hypertension",  "text": "Severe headache and dizziness. BP 180/110 hypertensive crisis. Emergency antihypertensive medication given."},
    {"doc_id": "D008", "category": "general",       "text": "Fever 101.3F, cough, fatigue for 5 days. Rapid flu test positive. Prescribed Tamiflu and rest."}
]

df = pd.DataFrame(documents)
df.to_csv("data/clinical_documents.csv", index=False)
print(f"✅ Created {len(documents)} documents")
print(df[["doc_id", "category"]].to_string(index=False))

✅ Created 8 documents
doc_id     category
  D001     diabetes
  D002   cardiology
  D003 hypertension
  D004     diabetes
  D005      general
  D006   cardiology
  D007 hypertension
  D008      general


In [16]:
from sentence_transformers import SentenceTransformer

# Load model
print("Loading model...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Model loaded!")

# Load documents
df = pd.read_csv("data/clinical_documents.csv")
documents  = df["text"].tolist()
doc_ids    = df["doc_id"].tolist()
categories = df["category"].tolist()

# Generate embeddings — show_progress_bar=False fixes GitHub issue
embeddings = model.encode(documents, show_progress_bar=False)

print(f"✅ Embeddings generated!")
print(f"   Documents : {embeddings.shape[0]}")
print(f"   Dimensions: {embeddings.shape[1]}")

np.save("output/embeddings.npy", embeddings)
print("💾 Saved to output/embeddings.npy")

Loading model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Model loaded!
✅ Embeddings generated!
   Documents : 8
   Dimensions: 384
💾 Saved to output/embeddings.npy


In [17]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def search(query, top_k=3):
    query_emb = model.encode([query], show_progress_bar=False)[0]
    scores = []
    for i, emb in enumerate(embeddings):
        scores.append({
            "rank":     0,
            "doc_id":   doc_ids[i],
            "category": categories[i],
            "score":    round(float(cosine_similarity(query_emb, emb)), 4),
            "snippet":  documents[i][:100] + "..."
        })
    scores = sorted(scores, key=lambda x: x["score"], reverse=True)[:top_k]
    for i, s in enumerate(scores):
        s["rank"] = i + 1
    return scores

print("✅ Search engine ready!")

✅ Search engine ready!


In [18]:
queries = [
    "diabetes medication and blood sugar",
    "chest pain and heart problems",
    "high blood pressure treatment",
    "annual health checkup",
    "surgery recovery care"
]

all_results = []

for query in queries:
    print(f"\n{'='*60}")
    print(f"🔍 {query}")
    print(f"{'='*60}")
    results = search(query, top_k=2)
    for r in results:
        print(f"  #{r['rank']} [{r['doc_id']}] Score: {r['score']} | {r['category']}")
        print(f"      {r['snippet']}")
        all_results.append({"query": query, **r})

print("\n✅ All searches complete!")


🔍 diabetes medication and blood sugar
  #1 [D001] Score: 0.5386 | diabetes
      Patient diagnosed with Type 2 diabetes. Prescribed metformin 500mg twice daily. Blood sugar at 180 m...
  #2 [D004] Score: 0.452 | diabetes
      Follow-up for diabetes. HbA1c improved from 9.2 to 7.8. Added insulin glargine for better glucose co...

🔍 chest pain and heart problems
  #1 [D002] Score: 0.6862 | cardiology
      Patient reports chest pain and shortness of breath. ECG shows irregular heartbeat. Referred to cardi...
  #2 [D006] Score: 0.3286 | cardiology
      Post cardiac surgery follow-up. Wound healing well. Patient advised to continue blood thinners and c...

🔍 high blood pressure treatment
  #1 [D003] Score: 0.5381 | hypertension
      Blood pressure 145/92 mmHg. Stage 2 hypertension. Prescribed lisinopril 10mg and low sodium diet....
  #2 [D007] Score: 0.4577 | hypertension
      Severe headache and dizziness. BP 180/110 hypertensive crisis. Emergency antihypertensive medication...

🔍 an

In [19]:
results_df = pd.DataFrame(all_results)
results_df.to_csv("output/search_results.csv", index=False)

print("=" * 45)
print("       RAG PIPELINE SUMMARY")
print("=" * 45)
print(f"  Documents indexed   : {len(documents)}")
print(f"  Embedding dimensions: {embeddings.shape[1]}")
print(f"  Queries run         : {len(queries)}")
print(f"  Results saved       : {len(all_results)}")
print("=" * 45)
print("  Output files:")
print("  → output/embeddings.npy")
print("  → output/search_results.csv")
print("=" * 45)
print("\n🎉 RAG Pipeline Complete!")

       RAG PIPELINE SUMMARY
  Documents indexed   : 8
  Embedding dimensions: 384
  Queries run         : 5
  Results saved       : 10
  Output files:
  → output/embeddings.npy
  → output/search_results.csv

🎉 RAG Pipeline Complete!
